# AQI Feature Store — Exploratory Data Analysis

Covers all 10 cities in `config.CITIES` over the full backfilled history (currently ~365 days). Run after `python -m feature_pipeline.backfill_pipeline` has populated the Hopsworks feature group, or via `.github/workflows/run_eda.yml` (executes headlessly, uploads this notebook with real outputs as a workflow artifact).

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from hopsworks_utils import get_feature_group

fg = get_feature_group()
df = fg.read()
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)

city_order = list(config.CITIES.keys())
city_labels = {k: v['label'] for k, v in config.CITIES.items()}

# The feature group can carry rows for cities retired from config.CITIES
# (e.g. Sukkur, replaced by Rawalpindi) - pipelines already ignore these
# since they only ever loop over config.CITIES, but this notebook reads
# the raw feature group directly, so drop them explicitly or every pooled
# stat below silently includes a city that's no longer part of the project.
dropped_cities = sorted(set(df['city'].unique()) - set(city_order))
if dropped_cities:
    print(f'Dropping rows for retired cities not in config.CITIES: {dropped_cities}')
df = df[df['city'].isin(city_order)].sort_values(['city', 'event_time']).reset_index(drop=True)

print(f'{len(df)} rows across {df["city"].nunique()} cities, '
      f'{df["event_time"].min()} to {df["event_time"].max()}')

## Missing values

By column overall, then by city — a city with a much higher null rate than the rest usually means a data-quality problem specific to it (this is exactly how the Sukkur → Rawalpindi swap got flagged).

In [ ]:
df.isna().mean().sort_values(ascending=False)

In [ ]:
missing_by_city = df.groupby('city').apply(lambda g: g['us_aqi'].isna().mean())
missing_by_city = missing_by_city.reindex(city_order).rename(index=city_labels)
missing_by_city.sort_values(ascending=False)

## Which cities have the worst air quality on average?

Ranks cities by mean/median/max US AQI over the full backfilled window — the single most direct answer to "which city is more polluted."

In [ ]:
city_stats = (
    df.groupby('city')['us_aqi']
    .agg(['mean', 'median', 'std', 'max'])
    .reindex(city_order)
    .rename(index=city_labels)
    .sort_values('mean', ascending=False)
)
city_stats

## AQI distribution by city

Boxplot, ordered by median — shows both typical level and spread/volatility per city, not just the mean.

In [ ]:
order = city_stats.sort_values('median', ascending=False).index.tolist()
data_by_city = [df[df['city'].map(city_labels) == c]['us_aqi'].dropna() for c in order]

fig, ax = plt.subplots(figsize=(12, 5))
ax.boxplot(data_by_city, tick_labels=order, showfliers=False)
ax.set_title('US AQI distribution by city (outliers hidden for readability)')
ax.set_ylabel('US AQI')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## AQI trend over time, per city

One panel per city so a spike in one city doesn't compress the y-axis for the rest.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 16), sharex=True)
for ax, city_key in zip(axes.flat, city_order):
    city_df = df[df['city'] == city_key]
    ax.plot(city_df['event_time'], city_df['us_aqi'], linewidth=0.7)
    ax.set_title(city_labels[city_key])
    ax.set_ylabel('US AQI')
plt.tight_layout()
plt.show()

## Seasonality: average AQI by hour of day (per city)

Heatmap — rows are cities, columns are hour-of-day (UTC), color is mean US AQI.

In [ ]:
def seasonality_heatmap(group_col, title, xlabel):
    pivot = (
        df.groupby(['city', group_col])['us_aqi']
        .mean()
        .unstack(group_col)
        .reindex(city_order)
        .rename(index=city_labels)
    )
    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label='Mean US AQI')
    plt.tight_layout()
    plt.show()
    print('Pooled (all cities) mean AQI by ' + xlabel.lower() + ', sorted worst first:')
    print(pivot.mean(axis=0).sort_values(ascending=False))
    return pivot

_ = seasonality_heatmap('hour', 'Average AQI by hour of day (UTC), per city', 'Hour')

## Seasonality: average AQI by month

Only meaningful now that the backfill covers a full year (a 90-day window couldn't show a full seasonal cycle).

In [ ]:
_ = seasonality_heatmap('month', 'Average AQI by month, per city', 'Month')

## Seasonality: average AQI by day of week

0 = Monday.

In [ ]:
_ = seasonality_heatmap('day_of_week', 'Average AQI by day of week, per city', 'Day of week (0=Mon)')

## Correlation between pollutants/weather and AQI

Pooled across all cities first, then per-city — a feature that correlates well overall but not in a specific city is a clue that city needs different features/handling.

In [ ]:
numeric_cols = df.select_dtypes('number').columns
corr = df[numeric_cols].corr()['us_aqi'].sort_values(ascending=False)
corr

In [ ]:
feature_cols = [c for c in numeric_cols if c != 'us_aqi']
per_city_corr = (
    df.groupby('city')[list(numeric_cols)]
    .apply(lambda g: g.corr()['us_aqi'].drop('us_aqi'))
    .reindex(city_order)
    .rename(index=city_labels)
)
per_city_corr

## Key findings

_From the ~369-day backfill (Aug 2025 – Aug 2026) across all 10 cities. Numbers below are from the executed run — re-running against a longer history may shift them somewhat, but the ranking/relationships are expected to hold._

- **Missing data is negligible.** Every raw pollutant/weather column is 0% missing; the only gaps are `aqi_lag_48h` (0.8%) and `aqi_lag_24h` (0.5%) — exactly the first 24-48h of history each city needs before those lag features exist, not an ongoing data problem.
- **Faisalabad, Multan, and Lahore are the most polluted cities by a clear margin** (mean US AQI 153-166, i.e. solidly "Unhealthy"), well above Peshawar/Islamabad/Rawalpindi (~113-115, "Unhealthy for Sensitive Groups") and Abbottabad/Karachi/Hyderabad/Quetta (~86-99, mostly "Moderate").
- **Strong winter smog season, weak daily/weekly pattern.** Pooled across all cities, January (mean AQI 154) and December (152) are by far the worst months, dropping to April (89, the cleanest) and September (97) - a ~65-point seasonal swing consistent with the regional winter smog/temperature-inversion pattern. Hour-of-day only swings ~15 points (worst ~13:00-14:00 UTC, ~130; calmest overnight, ~115), and day-of-week barely matters at all (Saturday 119 vs Tuesday 117 - a ~2-point range). Note this doesn't contradict the near-zero linear correlation for `month` two bullets down: month wraps around (Dec=12, Jan=1 are numerically far apart despite being adjacent seasonally), so a cyclical effect this strong can still show near-zero *linear* correlation against the raw 1-12 label - the groupby/heatmap above is the correct way to see it, not the correlation table.
- **AQI volatility predicts model performance.** Cities with higher AQI variance (Faisalabad std=54, Multan=49, Lahore=49) are exactly the same cities where every trained model (including the LSTM) posts negative R2 - across all 10 cities, the correlation between a city's AQI standard deviation and its average deployed-model R2 is **r=-0.64 (p=0.046)**. The forecasting difficulty isn't random or a modeling gap - it tracks a measurable property of each city's data.
- **Strongest predictors of AQI are its own recent history**, not external weather: `aqi_roll_mean_3h` (r=0.995), `aqi_roll_mean_24h` (r=0.941), `aqi_lag_24h` (r=0.870) dominate. Of the raw pollutants, pm2_5 is the strongest single predictor (r=0.755). Weather variables are weak by comparison - wind_speed_10m is the strongest of them at only r=-0.255 (higher wind disperses pollutants, as expected physically) - and calendar features (hour, day_of_week, month) barely correlate with raw AQI level at all (all |r|<0.07), which is exactly why the persistence baseline and lag/rolling features carry so much of the trained models' predictive power.
